# TREV Autograd Chemistry Benchmark: H2 & H4 (PUCCD)

Benchmarks autograd gradient on molecular chemistry problems with full IXYZ Hamiltonians.

**Molecules:** H2 (4 qubits), H4 (8 qubits) — STO-3G basis, Jordan-Wigner mapping

**Ansatz:** PUCCD (pair unitary coupled cluster doubles)

**Comparisons:**
1. Accuracy: autograd (f64) vs parameter-shift (RIGHT_SUFFIX_SAMPLING with QWC)
2. Speed: autograd vs parameter-shift gradient timing
3. VQE convergence to exact ground state energy

In [ ]:
!pip install -q git+https://github.com/keunjunpark/TREV.git@autograd_qr_decomp --force-reinstall --no-deps
!pip install -q pyscf qiskit qiskit-nature

In [ ]:
import torch, time, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.sparse.linalg as spla

from TREV.circuit import Circuit
from TREV.hamiltonian.hamiltonian import Hamiltonian
from TREV.measure.enums import MeasureMethod
from TREV.optimization.gradients.autograd_gradient import autograd_gradient
from TREV.optimization.gradients.batch_parameter_shift import batch_gradient

from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit_nature.second_q.circuit.library import PUCCD, HartreeFock
from qiskit.compiler import transpile as qiskit_transpile
from qiskit.circuit import ParameterExpression

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    free_b, total_b = torch.cuda.mem_get_info(device)
    print(f'GPU memory: {free_b/1e9:.1f} GB free / {total_b/1e9:.1f} GB total')
print(f'PyTorch: {torch.__version__}')

## Setup: Molecules & Circuit Builder

In [ ]:
jw_mapper = JordanWignerMapper()
BASIS_GATES = ['h', 'x', 'cx', 'rz', 'rx']

MOLECULES = {
    'H2':  {'atom': 'H 0 0 0; H 0 0 0.735', 'charge': 0, 'spin': 0},
    'H4':  {'atom': 'H 0 0 0; H 0 0 0.735; H 0 0 1.47; H 0 0 2.205',
            'charge': 0, 'spin': 0},
}

def setup_molecule(name):
    """Create molecular Hamiltonian + exact ground state energy."""
    mol = MOLECULES[name]
    driver = PySCFDriver(atom=mol['atom'], charge=mol['charge'],
                         spin=mol['spin'], basis='sto3g')
    problem = driver.run()
    ns = problem.num_spatial_orbitals
    np_ = problem.num_particles
    N = 2 * ns
    nuc_rep = problem.nuclear_repulsion_energy
    qubitOp = jw_mapper.map(problem.hamiltonian.second_q_op())

    mat = qubitOp.to_matrix(sparse=True)
    eigvals, _ = spla.eigsh(mat, k=1, which='SA')
    exact = float(eigvals[0]) + nuc_rep

    ps, cs = [], []
    for elm in qubitOp:
        ps.append(str(elm.paulis[0][::-1]))
        cs.append(float(elm.coeffs[0].real))
    h = Hamiltonian(N, ps, cs)

    print(f"  {name}: {N} qubits, {len(h.paulis)} Hamiltonian terms, "
          f"QWC groups={len(h.get_qwc_groups())}")
    print(f"    E_exact = {exact:.8f} Ha, has_only_zi = {h.has_only_zi}")
    return h, N, ns, np_, nuc_rep, exact


def build_puccd(name, ns, np_, N, chi):
    """Build PUCCD circuit transpiled to TREV gates."""
    hf = HartreeFock(ns, np_, jw_mapper)
    ansatz = PUCCD(ns, np_, jw_mapper, initial_state=hf)
    transpiled = qiskit_transpile(ansatz, basis_gates=BASIS_GATES, optimization_level=0)

    c = Circuit(N, device=device)
    fixed_idx, fixed_vals, var_idx = [], [], []
    for inst in transpiled.data:
        gate = inst.operation
        qs = [transpiled.find_bit(q).index for q in inst.qubits]
        if gate.name == 'h': c.h(qs[0])
        elif gate.name == 'x': c.x(qs[0])
        elif gate.name == 'cx': c.cx(qs[0], qs[1])
        elif gate.name in ('rz', 'rx'):
            pidx = c.params_size
            if gate.name == 'rz': c.rz(qs[0])
            else: c.rx(qs[0])
            angle = gate.params[0]
            if isinstance(angle, ParameterExpression): var_idx.append(pidx)
            else: fixed_idx.append(pidx); fixed_vals.append(float(angle))
        elif gate.name in ('id', 'barrier'): pass
    c.rank = chi
    print(f"    PUCCD chi={chi}: {c.params_size} params "
          f"({len(var_idx)} var, {len(fixed_idx)} fixed)")
    return c, fixed_idx, fixed_vals, var_idx


def init_theta(c, fixed_idx, fixed_vals, var_idx, seed=42):
    """Initialize theta with fixed params + small random variational."""
    theta = torch.zeros(c.params_size, device=device)
    if fixed_idx:
        theta[torch.tensor(fixed_idx, device=device)] = \
            torch.tensor(fixed_vals, device=device)
    if var_idx:
        torch.manual_seed(seed)
        theta[torch.tensor(var_idx, device=device)] = \
            torch.randn(len(var_idx), device=device) * 0.01
    return theta


def bench(fn, warmup=3, repeats=5):
    for _ in range(warmup): fn()
    if device == 'cuda': torch.cuda.synchronize()
    times = []
    for _ in range(repeats):
        if device == 'cuda': torch.cuda.synchronize()
        t0 = time.perf_counter()
        fn()
        if device == 'cuda': torch.cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1000)
    return np.median(times)


# Setup molecules
mol_data = {}
for name in ['H2', 'H4']:
    mol_data[name] = setup_molecule(name)

## 1. Gradient Accuracy: Autograd vs Parameter-Shift (RSS)

Compare autograd (complex128, exact) vs parameter-shift with RIGHT_SUFFIX_SAMPLING
(QWC groups, supports IXYZ) on PUCCD circuits.

In [ ]:
torch.manual_seed(42)
rows = []

configs = [
    ('H2', 4), ('H2', 8), ('H2', 16),
    ('H4', 4), ('H4', 8), ('H4', 16),
]

print(f"{'config':<30} {'P':>4} {'T':>5} | {'cos(ad,rss)':>12} {'max|ad-rss|':>12}")
print("-" * 75)

for mol_name, chi in configs:
    torch.cuda.empty_cache()
    h, N, ns, np_, nuc_rep, exact = mol_data[mol_name]
    c, fi, fv, vi = build_puccd(mol_name, ns, np_, N, chi)
    theta = init_theta(c, fi, fv, vi)
    T = len(h.paulis)

    # Autograd (complex128 for deep circuit stability)
    ga, ev_ad, _ = autograd_gradient(theta, c, h, torch.complex128)

    # Parameter-shift with RSS (supports IXYZ via QWC)
    gp = batch_gradient(theta, c, h, 8, 50000, np.pi/2, 1, 0, False,
                        MeasureMethod.RIGHT_SUFFIX_SAMPLING)

    cos_val = torch.nn.functional.cosine_similarity(
        ga.unsqueeze(0), gp.unsqueeze(0)).item()
    maxerr = (ga - gp).abs().max().item()

    label = f"{mol_name} PUCCD N={N} chi={chi}"
    print(f"{label:<30} {c.params_size:>4} {T:>5} | {cos_val:>12.6f} {maxerr:>12.2e}")
    rows.append({'mol': mol_name, 'N': N, 'chi': chi, 'P': c.params_size,
                 'T': T, 'cos': cos_val, 'max_err': maxerr})

df_acc = pd.DataFrame(rows)
df_acc

## 2. Gradient Speed

Autograd (exact, 1 fwd+bwd) vs parameter-shift RSS (sampling, 2P evaluations).

In [ ]:
torch.manual_seed(42)
rows_speed = []

speed_configs = [
    ('H2', 4), ('H2', 8), ('H2', 16),
    ('H4', 4), ('H4', 8),
]

print(f"{'config':<30} {'P':>4} {'T':>5} | {'autograd':>10} {'PS-RSS':>10} {'speedup':>8}")
print("-" * 75)

for mol_name, chi in speed_configs:
    torch.cuda.empty_cache()
    h, N, ns, np_, nuc_rep, exact = mol_data[mol_name]
    c, fi, fv, vi = build_puccd(mol_name, ns, np_, N, chi)
    theta = init_theta(c, fi, fv, vi)
    T = len(h.paulis)

    def run_ad():
        autograd_gradient(theta, c, h, torch.complex128)
    def run_ps():
        batch_gradient(theta, c, h, 8, 10000, np.pi/2, 1, 0, False,
                       MeasureMethod.RIGHT_SUFFIX_SAMPLING)

    ms_ad = bench(run_ad)
    ms_ps = bench(run_ps)
    sp = ms_ps / ms_ad

    label = f"{mol_name} PUCCD chi={chi}"
    print(f"{label:<30} {c.params_size:>4} {T:>5} | {ms_ad:>8.1f}ms {ms_ps:>8.1f}ms {sp:>7.2f}x")
    rows_speed.append({'mol': mol_name, 'N': N, 'chi': chi, 'P': c.params_size,
                       'T': T, 'autograd_ms': ms_ad, 'ps_rss_ms': ms_ps, 'speedup': sp})

df_speed = pd.DataFrame(rows_speed)
df_speed

## 3. VQE Convergence: Autograd vs Parameter-Shift

Full optimization on H2 and H4 with PUCCD ansatz. Compare convergence to exact ground state.

In [ ]:
from TREV.measure.right_suffix_sampling import expectation_value as rss_ev

torch.manual_seed(42)
CHEM_ACCURACY = 1.6e-3  # Ha
iters = 300

vqe_configs = [('H2', 8), ('H4', 8)]
fig, axes = plt.subplots(1, len(vqe_configs), figsize=(7*len(vqe_configs), 5))
if len(vqe_configs) == 1: axes = [axes]

for idx, (mol_name, chi) in enumerate(vqe_configs):
    torch.cuda.empty_cache()
    h, N, ns, np_, nuc_rep, exact = mol_data[mol_name]
    c, fi, fv, vi = build_puccd(mol_name, ns, np_, N, chi)
    theta_init = init_theta(c, fi, fv, vi)
    fi_t = torch.tensor(fi, device=device) if fi else None
    fv_dev = torch.tensor(fv, device=device) if fv else None

    beta1, beta2, eps_adam, lr = 0.9, 0.999, 1e-8, 0.005

    # ---- Autograd VQE (exact, complex128) ----
    theta = theta_init.clone()
    m, v = torch.zeros_like(theta), torch.zeros_like(theta)
    energies_ad = []

    t0 = time.time()
    for step in range(iters):
        grad, ev, _ = autograd_gradient(theta, c, h, torch.complex128)
        if fi_t is not None: grad[fi_t] = 0.0
        m = beta1*m + (1-beta1)*grad
        v = beta2*v + (1-beta2)*grad**2
        mh = m / (1 - beta1**(step+1))
        vh = v / (1 - beta2**(step+1))
        theta = theta - lr * mh / (vh.sqrt() + eps_adam)
        if fi_t is not None and fv_dev is not None: theta[fi_t] = fv_dev
        energies_ad.append(ev + nuc_rep)
        if step % 100 == 0 or step == iters-1:
            print(f"  [AD {mol_name}] step {step:>3}: E={energies_ad[-1]:.6f} (err={energies_ad[-1]-exact:+.6f})")
    t_ad = time.time() - t0

    # ---- Param-shift RSS VQE (sampling) ----
    theta = theta_init.clone()
    m, v = torch.zeros_like(theta), torch.zeros_like(theta)
    energies_ps = []

    t0 = time.time()
    for step in range(iters):
        grad = batch_gradient(theta, c, h, 8, 10000, np.pi/2, 1, 0, False,
                              MeasureMethod.RIGHT_SUFFIX_SAMPLING)
        if fi_t is not None: grad[fi_t] = 0.0
        m = beta1*m + (1-beta1)*grad
        v = beta2*v + (1-beta2)*grad**2
        mh = m / (1 - beta1**(step+1))
        vh = v / (1 - beta2**(step+1))
        theta = theta - lr * mh / (vh.sqrt() + eps_adam)
        if fi_t is not None and fv_dev is not None: theta[fi_t] = fv_dev
        cores = [c.build_tensor(theta)[i] for i in range(N)]
        ev_ps = rss_ev(cores, h, shots=10000, seed=step)
        energies_ps.append(ev_ps + nuc_rep)
        if step % 100 == 0 or step == iters-1:
            print(f"  [PS {mol_name}] step {step:>3}: E={energies_ps[-1]:.6f} (err={energies_ps[-1]-exact:+.6f})")
    t_ps = time.time() - t0

    # Plot
    ax = axes[idx]
    ax.plot(energies_ad, label=f'Autograd ({t_ad:.0f}s, {t_ad/iters*1000:.0f}ms/it)', linewidth=2)
    ax.plot(energies_ps, '--', label=f'PS-RSS ({t_ps:.0f}s, {t_ps/iters*1000:.0f}ms/it)',
            linewidth=2, alpha=0.8)
    ax.axhline(exact, ls=':', color='black', lw=1.5, label=f'Exact ({exact:.4f} Ha)')
    ax.axhline(exact + CHEM_ACCURACY, ls='--', color='gray', lw=1, alpha=0.5,
               label='Chem. accuracy')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Energy (Ha)')
    sp = t_ps / t_ad
    ax.set_title(f'{mol_name} PUCCD (N={N}, chi={chi}, P={c.params_size})\n'
                 f'Autograd {sp:.1f}x faster')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    err_ad = energies_ad[-1] - exact
    err_ps = energies_ps[-1] - exact
    print(f"\n{mol_name} chi={chi}: ad={t_ad/iters*1000:.0f}ms/it  ps={t_ps/iters*1000:.0f}ms/it  "
          f"speedup={sp:.2f}x")
    print(f"  E_ad={energies_ad[-1]:.6f}  E_ps={energies_ps[-1]:.6f}  "
          f"E_exact={exact:.6f}")
    print(f"  err_ad={err_ad:+.6f}  err_ps={err_ps:+.6f}  "
          f"chem_acc={'YES' if abs(err_ad)<CHEM_ACCURACY else 'no'} / "
          f"{'YES' if abs(err_ps)<CHEM_ACCURACY else 'no'}\n")

plt.tight_layout()
plt.show()

## 4. SVD Stability Scaling

Test gradient accuracy vs circuit depth and chi to understand limits.
Deep circuits need complex128 + the deterministic noise fix.

In [ ]:
from TREV.optimization.gradients.autograd_gradient import _build_tensor_diff, _contraction_diff_vectorized

h_test = Hamiltonian(num_qubits=4)
h_test.add_pauli('ZZII', 0.3); h_test.add_pauli('XXII', 0.2)
h_test.add_pauli('YYII', -0.1); h_test.add_pauli('IIII', 1.0)

rows_svd = []

print(f"{'depth':>6} {'#CX':>5} {'chi':>4} | {'cos f64':>10} {'cos f32':>10}")
print("-" * 45)

for depth in [1, 3, 5, 10, 20, 30]:
    for chi in [4, 8]:
        N = 4
        c = Circuit(N, device=device)
        for i in range(N): c.h(i)
        n_cx = 0
        for _ in range(depth):
            for i in range(N-1): c.cx(i, i+1); n_cx += 1
            for i in range(N): c.ry(i); c.rz(i)
        c.rank = chi

        torch.manual_seed(42)
        theta = torch.randn(c.params_size, device=device)

        results = {}
        for label, dtype in [('f64', torch.complex128), ('f32', torch.cfloat)]:
            rdtype = torch.float64 if dtype == torch.complex128 else torch.float32
            ga, _, _ = autograd_gradient(theta, c, h_test, dtype)
            eps = 1e-4
            gf = torch.zeros(theta.numel(), device=device, dtype=rdtype)
            for k in range(theta.numel()):
                tp = theta.to(rdtype).clone(); tp[k] += eps
                fp = _contraction_diff_vectorized(_build_tensor_diff(tp, c, dtype), h_test, dtype).item()
                tp = theta.to(rdtype).clone(); tp[k] -= eps
                fm = _contraction_diff_vectorized(_build_tensor_diff(tp, c, dtype), h_test, dtype).item()
                gf[k] = (fp - fm) / (2*eps)
            if gf.abs().max() < 1e-10:
                results[label] = float('nan')
            else:
                results[label] = torch.nn.functional.cosine_similarity(
                    ga.unsqueeze(0).to(rdtype), gf.unsqueeze(0)).item()

        print(f"{depth:>6} {n_cx:>5} {chi:>4} | {results['f64']:>10.6f} {results['f32']:>10.6f}")
        rows_svd.append({'depth': depth, 'n_cx': n_cx, 'chi': chi,
                         'cos_f64': results['f64'], 'cos_f32': results['f32']})

df_svd = pd.DataFrame(rows_svd)

# Plot
fig, ax = plt.subplots(figsize=(8, 5))
for chi_val in df_svd['chi'].unique():
    sub = df_svd[df_svd['chi'] == chi_val]
    ax.plot(sub['n_cx'], sub['cos_f64'], 'o-', label=f'f64 chi={chi_val}', linewidth=2)
    ax.plot(sub['n_cx'], sub['cos_f32'], 's--', label=f'f32 chi={chi_val}', alpha=0.6)
ax.axhline(0.99, ls=':', color='gray', alpha=0.5, label='cos=0.99')
ax.set_xlabel('Number of CX gates (SVDs)')
ax.set_ylabel('cos(autograd, FD)')
ax.set_title('SVD Backward Stability vs Circuit Depth')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_ylim(-0.5, 1.05)
plt.tight_layout()
plt.show()

## Summary

| Feature | Autograd | Param-shift (RSS) |
|---------|----------|------------------|
| IXYZ support | vectorized contraction | QWC groups |
| Gradient cost | 1 fwd + 1 bwd | 2P evaluations |
| Exact / sampling | exact | sampling (needs shots) |
| Deep circuit stability | needs complex128 + noise | always stable |
| Chemistry VQE | fast, exact gradients | slower, noisy gradients |